# 스캔 PDF 페이지 분류 PoC — Colab 실행

위에서부터 차례로 실행하세요 (**런타임 → 모두 실행**). GPU(T4)가 있으면 빠르지만, 없어도 돌아갑니다.

샘플은 전부 가짜 서류입니다. **실제 고객 서류는 이 노트북에 올리지 마세요** (구글 서버에 올라갑니다).

In [ ]:
%cd /content
!rm -rf scan-pdf-sorter-poc
!git clone -q https://github.com/weriousdf/scan-pdf-sorter-poc.git
%cd scan-pdf-sorter-poc
!pip install -q -r requirements.txt

In [ ]:
import torch, transformers, pymupdf
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 (CPU)')
print('torch', torch.__version__, '| transformers', transformers.__version__, '| pymupdf', pymupdf.__version__)

## 1. 기준선 — 규칙 (잉크 영역 크기)

In [ ]:
!python poc/split_pdf.py --pdfs data/samples --backend rule --run-id colab_run1
!python poc/score.py --truth data/ground_truth.csv --pred outputs/colab_run1/rule/predictions.csv --sweep

## 2. AI — CLIP 제로샷 분류 (설명문 v1: 결과를 보기 전에 작성)
처음 실행할 때 모델(약 600MB)을 내려받습니다.

In [ ]:
!python poc/split_pdf.py --pdfs data/samples --backend clip --clip-prompts v1 --run-id colab_run1
!python poc/score.py --truth data/ground_truth.csv --pred outputs/colab_run1/clip/predictions.csv --sweep

## 3. AI — CLIP 설명문 v2 (v1 의 가짜 샘플 실패만 보고 보강)

In [ ]:
!python poc/split_pdf.py --pdfs data/samples --backend clip --clip-prompts v2 --run-id colab_run2
!python poc/score.py --truth data/ground_truth.csv --pred outputs/colab_run2/clip/predictions.csv --sweep

In [ ]:
import csv, matplotlib.pyplot as plt
from PIL import Image
truth = {(r['pdf'], r['page']): r for r in csv.DictReader(open('data/ground_truth.csv', encoding='utf-8-sig'))}
preds = list(csv.DictReader(open('outputs/colab_run2/clip/predictions.csv', encoding='utf-8-sig')))
fig, axes = plt.subplots(3, 7, figsize=(21, 10))
for ax in axes.flat: ax.axis('off')
for ax, p in zip(axes.flat, preds):
    t = truth[(p['pdf'], p['page'])]
    ok = t['size'] == p['size'] and t['orientation'] == p['orientation']
    ax.imshow(Image.open('outputs/colab_run2/clip/' + p['file']))
    title = p['pdf'][5:-4] + ' p' + p['page'] + ' | ' + p['size'] + '/' + p['orientation'] + ('' if ok else ' [X]')
    ax.set_title(title, fontsize=9, color='green' if ok else 'red')
plt.tight_layout(); plt.show()

## 4. 결과 내려받기

In [ ]:
!zip -qr outputs_colab.zip outputs
from google.colab import files
files.download('outputs_colab.zip')